Imports

In [2]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
openai = OpenAI()

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-;")
else:
    print("API key found!")


headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}


API key found!


Getting llama 3.2

In [3]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


Setting Llama

In [4]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


Response for a fun fact

In [4]:
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Here's one:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's unique combination of acidity, water content, and its low oxygen exposure help to prevent the growth of bacteria and other microorganisms that can cause it to spoil. Isn't that sweet?"

Deepseek

In [7]:
!ollama pull deepseek-r1:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏  50 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 657 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 1.7 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 2.8 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 3.6 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 4.6 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 6.4 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 6.4 MB/1

In [8]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Sure! Here\'s a fun fact: \n\nThe story of the _______ is famous. (Replace __ with any name like "Abraham")'

Exercise - Website summarize

In [ ]:
import sys
from pathlib import Path

src_path = Path().resolve().parent / "src"
sys.path.append(str(src_path))

from scraper import fetch_website_contents
from IPython.display import Markdown, display

system_prompt = """
You are a assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://anthropic.com")
# display_summary("https://openai.com/")

*This is a company website run by Anthropic, a public benefit corporation that aims to secure the benefits of AI and mitigate its risks.*

**News/Announcements**

* Anthropic completed the first AI-planned drive on Mars, a "Four Hundred Meters" project. Read about it [here](Read the story)
* Anthropic launched Claude Opus 4.5, an advanced model for coding, agents, computer use, and enterprise workflows.

**Main Idea**: A company dedicated to making human well-being safer with AI technology.

In [10]:
from IPython.display import Markdown, display

system_prompt = """
Your task is to read an email and suggest an appropriate short subject line for the email and a resume of the context.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a email.
Provide a short subject line for the email and a resume of the context.

"""

def messages_for(content):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + content}
    ]

email_content = """
Subject: Your Car Rental Confirmation – SilverWay Rentals

Dear Customer,

Thank you for choosing SilverWay Car Rentals for your upcoming trip. We’re pleased to confirm your reservation and look forward to providing you with a smooth and comfortable rental experience.

Reservation Details:
- Reservation Number: SWR-458239
- Pickup Location: SilverWay Downtown Branch
- Pickup Date & Time: March 14, 2026 at 10:00 AM
- Return Date & Time: March 18, 2026 at 6:00 PM
- Vehicle Type: Toyota Corolla or similar (Compact Class)
- Add-ons: GPS Navigation, Full Coverage Insurance

Please remember to bring a valid driver’s license, a credit card in the main driver’s name, and your reservation number at the time of pickup. Our team will be happy to assist you with any questions when you arrive.

If you need to modify or cancel your reservation, you can do so by replying to this email or contacting our support team at support@silverwayrentals.com at least 24 hours before your scheduled pickup time.

We appreciate your trust in SilverWay Car Rentals and wish you a safe and enjoyable journey.

Kind regards,
Emily Carter
Customer Experience Team
SilverWay Car Rentals
www.silverwayrentals.com
+1 (555) 013-4728
"""

def summarize_email(email_content):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages = messages_for(email_content)
    )
    return response.choices[0].message.content

def display_summary_email(content):
    summary = summarize_email(content)
    display(Markdown(summary))

display_summary_email(email_content)

Subject: Confirmation of Car Rental Reservation with SilverWay Rentals

This email is sent to customers who have made a car rental reservation with SilverWay Rentals. The company confirms the reservation details, including vehicle type and add-ons, and provides instructions on what to bring at pickup and how to modify or cancel the reservation if needed.